# Plot fluc blue (figure m05)

In [ ]:
# Libraries
import numpy as np
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
# Directories
dir_nearth = '../data/naturalearth/'
dir_zen ='../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

In [ ]:
# Get cspatial carbon density data
ds_in = xr.open_dataset(f'{dir_zen}ecosystem_carbon_data_figure_m05.nc')

# Get coastline data
# Download: https://www.naturalearthdata.com/
coastline110 = gpd.read_file(
    f'{dir_nearth}ne_110m_coastline/ne_110m_coastline.shp')

---

### Data preparation

In [ ]:
# Dataset with yearly global values
ds_ysum = (
    (ds_in[[v for v in ds_in.data_vars if '_fluc_' in v]] * ds_in.cell_area)
        .sum(['lat', 'lon']) * 1E-9
)

# Dataset with global values
ds_sum = (
    (ds_in[[v for v in ds_in.data_vars if '_fluc_' in v]] * ds_in.cell_area)
        .sum(['time', 'lat', 'lon']) * 1E-9
)

# Land-sea mask
da_land = ds_in.chansis_fluc_net[0].notnull()

# Dictionary with pft labels
def get_pft_dict():
    return {
        int(var.split('pft')[-1]): ds_in[var].attrs['pft']
        for var in ds_in.data_vars
        if "chansis_fluc_pft" in var
    }

pft_dict = get_pft_dict()

---

### Subplot functions

In [ ]:
def subplot_line_net(ax):
    
    """Subplot fluc lineplot."""
    
    # Plot
    for c, ls in zip(['chansis', 'cspatial', 'cdgvm'], 
                     [ls_chansis, ls_cspatial, ls_cdgvm]):
        ds_ysum[f'{c}_fluc_net'].plot(ax=ax, c=c_net, ls=ls)

    # Layout
    ax.set_ylim(0, 2.3)
    ax.set_xlim(xmin, xmax+1)
    ax.axhline(0, color='#000000')
    ax.set_ylabel(r'f$_{LUC}$ (PgC yr$^{-1}$)')
    ax.set_xlabel(None)
    ax.spines[['right', 'top']].set_visible(False)
    
    # Legend
    handles_fluc = [
        Line2D([0], [0], c='#000000', ls=ls_chansis, label=label_chansis),
        Line2D([0], [0], c='#000000', ls=ls_cspatial, label=label_cspatial),
        Line2D([0], [0], c='#000000', ls=ls_cdgvm, label=label_cdgvm),
    ]
    
    ax.legend(handles=handles_fluc, loc='lower left', ncol=1, 
              facecolor='white', edgecolor='black', framealpha=0.8)

In [ ]:
def subplot_map(ax):
    
    """Subplot fluc difference cspatial vs. chansis map."""
    
    im = ((ds_in.cspatial_fluc_net - ds_in.chansis_fluc_net).sum('time')
        .where(da_land)
        .plot.imshow(ax=ax, vmin=-20, vmax=20, cmap='PuOr', add_colorbar=False)
         )
    
    coastline110.plot(ax=ax, color='#000000', linewidth=0.5)
    
    ax.set_title(None)
    ax.axis('off')
    ax.set_ylim(-57, 90)

    cbar = fig.colorbar(
        im, ax=ax,
        orientation='horizontal', pad=0.08,
        label=r'Cumulative f$_{LUC}$ 1960–2023 (tC ha$^{-1}$)',
        aspect=50, shrink=0.8, extend="both"
    )
    
    cbar.ax.text(
        0.0, 2.2, 'BLUE cHansis higher',
        transform=cbar.ax.transAxes,
        ha='left', va='center')
    
    cbar.ax.text(
        1.0, 2.2, 'BLUE cSpatial higher',
        transform=cbar.ax.transAxes,
        ha='right', va='center')

In [ ]:
def subplot_line_components(ax):
    
    """Subplot component lineplot."""
    
    # Plot
    for v, c in zip(('deforestation', 
                     'forest_regrowth', 
                     'wood_harvest', 
                     'other_transitions'),
                    (c_deforestation, c_regrowth, c_harvest, c_other)):
        ds_ysum[f'chansis_fluc_{v}'].plot(ax=ax, c=c, ls=ls_chansis)
        ds_ysum[f'cspatial_fluc_{v}'].plot(ax=ax, c=c, ls=ls_cspatial)
        ds_ysum[f'cdgvm_fluc_{v}'].plot(ax=ax, c=c, ls=ls_cdgvm)        
        
    # Labels
    for label_y, label, c in zip(
        [label_deforestation_y, 
         label_regrowth_y, 
         label_harvest_y, 
         label_other_y],
        ["Deforestation", 
         "Forest (re-)growth", 
         "Wood harvest", 
         "Other transitions"],
        [c_deforestation, c_regrowth, c_harvest, c_other]
    ):
        ax.text(xmin + 0.5, label_y, 
                label, 
                ha='left', va='center', 
                fontsize=10, c=c)
    
    # Layout
    ax.set_ylim(-2.0, 2.5)
    ax.set_yticks(np.arange(-2, 2.6, 1))
    ax.set_xlim(xmin, xmax+1)
    ax.axhline(0, color='#000000')
    ax.set_ylabel(r'f$_{LUC}$ (PgC yr$^{-1}$)')
    ax.set_xlabel(None)
    ax.spines[['right', 'top']].set_visible(False)

In [ ]:
def subplot_bar_components(ax):
    
    """Subplot component barplot."""
    
    def add_bars(ax, c, position, xmin, xmax):
    
        def add_labels(ax, x, v, c, offset):    
            if v >= 0:
                v_offset, va = offset, 'bottom'
            else:
                v_offset, va = -offset, 'top'
                
            ax.text(x, v + v_offset, f'{v:.0f}', color=c, 
                    ha='center', va=va, fontsize='small')

        
        # Define values
        bar_width = .15
        
        v_net = ds_sum[f'{c}_fluc_net'].item()
        v_deforestation = ds_sum[f'{c}_fluc_deforestation'].item()
        v_regrowth = ds_sum[f'{c}_fluc_forest_regrowth'].item() 
        v_harvest = ds_sum[f'{c}_fluc_wood_harvest'].item()
        v_other = ds_sum[f'{c}_fluc_other_transitions'].item()
        
        x_net = position + bar_width*-2
        x_deforestation = position + bar_width*-1
        x_regrowth = position + bar_width*0
        x_harvest = position + bar_width*1
        x_other = position + bar_width*2

        # Plot
        bars = ax.bar(
            [x_net, x_deforestation, x_regrowth, x_harvest, x_other], 
            [v_net, v_deforestation, v_regrowth, v_harvest, v_other], 
            color=[c_net, c_deforestation, c_regrowth, c_harvest, c_other], 
            width=bar_width
        )
        
        # Layout
        ax.axhline(0, color='#000000')

        # Label
        add_labels(ax, x_net, v_net, c_net, 1)
        add_labels(ax, x_deforestation, v_deforestation, c_deforestation, 1)
        add_labels(ax, x_regrowth, v_regrowth, c_regrowth, 1)
        add_labels(ax, x_harvest, v_harvest, c_harvest, 1)
        add_labels(ax, x_other, v_other, c_other, 1)

        
    # Plot
    add_bars(ax, 'chansis', 0, xmin, xmax)
    add_bars(ax, 'cspatial', 1, xmin, xmax)
    add_bars(ax, 'cdgvm', 2, xmin, xmax)

    # Layout
    ax.set_ylim(-100, 150)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(
        ["\n".join(i.split(' ')) 
         for i in [label_chansis, label_cspatial, label_cdgvm]]
    )
    ax.tick_params(axis='x', length=0)
    ax.axvline(0.5, color='#000000', ls=':')
    ax.axvline(1.5, color='#000000', ls=':')
    ax.set_ylabel(r'Cumulative f$_{LUC}$ 1960–2023 (PgC)')
    ax.axhline(0, color='#000000')
    ax.spines[['right', 'top', 'bottom']].set_visible(False)

In [ ]:
def subplot_bar_pft(ax):
    
    """Subplot pft barplot with line-style bars."""

    # constants
    bar_height = 0.25
    l_pfts = list(range(1, 12))
    c_line = '#000000'
    
    # Configuration for each dataset
    l_plot_items = [
        ('chansis', 'BLUE cHansis', '-', -bar_height),
        ('cspatial', 'BLUE cSpatial', '--', 0),
        ('cdgvm', 'BLUE cDGVM', ':', bar_height),
    ]
        
    # Plot each dataset
    for c, label, linestyle, offset in l_plot_items:    
        for i in l_pfts:
            v = ds_sum[f'{c}_fluc_pft{i}']
            y_position = i + offset
            
            # Plot data as line and marker at the end
            ax.hlines(y_position, 0, v, colors=c_line, linestyles=linestyle,
                      label=label if i == 1 else "")
            ax.plot(v, y_position, marker='x', color=c_line)
    
    # Background shading
    for y0, y1 in zip(np.arange(1, 12, 2), np.arange(2, 13, 2)):
        ax.axhspan(y0-0.5, y1-0.5, color='black', alpha=0.05, zorder=0)
    
    # Styling
    ax.axvline(x=0, color='black', linestyle='-', alpha=0.7, linewidth=0.8)
    ax.grid(axis='x', linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    ax.spines[['right', 'top', 'bottom']].set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)
    
    # Labels and limits
    ax.set_xlabel(r'Cumulative f$_{LUC}$ 1960–2023 (PgC)')
    ax.set_yticks(list(l_pfts))
    ax.set_yticklabels(pft_dict.values())
    ax.set_ylim(0.5, 11.5)
    ax.set_xlim(0, None)
    ax.invert_yaxis()

---

### Plot

In [ ]:
# Plot labels and settings
label_chansis, label_cspatial, label_cdgvm = 'BLUE cHansis', 'BLUE cSpatial', 'BLUE cDGVM'
xmin, xmax = 1960, 2023
c_net = '#000000'
ls_chansis, ls_cspatial, ls_cdgvm = '-', '--', ':'
c_deforestation, c_regrowth, c_harvest, c_other = '#e41a1c', '#4daf4a', '#ff7f00', '#377eb8'
label_net_y, label_deforestation_y, label_regrowth_y, label_harvest_y, label_other_y = 2.35, 1.55, -1.45, 0.8, -0.2

In [ ]:
fig = plt.figure(figsize=(12, 12), dpi=100)

ax0 = fig.add_axes([0.0, 0.64, 0.39, 0.24])
ax1 = fig.add_axes([0.44, 0.61, 0.43, 0.39])
ax2 = fig.add_axes([0.0, 0.34, 0.39, 0.24])
ax3 = fig.add_axes([0.49, 0.34, 0.39, 0.24]) 
ax4 = fig.add_axes([0.22, 0.04, 0.66, 0.24]) 

# Plots
subplot_line_net(ax0)
subplot_map(ax1)
subplot_line_components(ax2)
subplot_bar_components(ax3)
subplot_bar_pft(ax4)

# Titles
fig.text(-0.045, 0.865, 'A', fontsize='xx-large', fontweight='bold')
fig.text(0.425, 0.865, 'B', fontsize='xx-large', fontweight='bold')
fig.text(-0.045, 0.575, 'C', fontsize='xx-large', fontweight='bold')
fig.text(0.425, 0.575, 'D', fontsize='xx-large', fontweight='bold')
fig.text(-0.045, 0.275, 'E', fontsize='xx-large', fontweight='bold')

# Export
fstr_out = 'fig_m05_fluc'
fig.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 
fig.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 